In [30]:
import pandas as pd
import numpy as np
import random
import warnings
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score

In [31]:
# Učitavanje podataka
df = pd.read_csv("iris.csv")
X = df.drop('species', axis=1)
y = df['species']
print(f"Dataset: {X.shape[0]} uzoraka, {X.shape[1]} feature-a")
X.head()

Dataset: 150 uzoraka, 4 feature-a


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## 1. RandomSearch (0.5 bodova)

Manuelna implemntacija RandomSearch-a koja nasumicno bira kombinacije hiperparametara (ukljucujuci `tt_split`) i evalira SVM model.

In [ ]:
# kao sa vjezbi samo tt split diff
OPCIJE = {
    "tt_split": np.arange(0.1, 0.5, 0.01).tolist(),       # koliko posto podataka ide u test set (10%-50%)
    "random_state": np.arange(0, 50).tolist(),              # seed za reproducibilnost rezultata
    "C": np.arange(0.1, 10.1, 0.1).tolist(),               # regularizacija - manji C = vise tolerise greske, veci C = strozi model
    "kernel": ["linear", "rbf"],                            # tip granice odlucivanja - linear = ravna linija, rbf = zakrivljena
    "gamma": ["scale", "auto"],                             # koliko daleko utice jedan uzorak (samo za rbf)
    "break_ties": [True, False],                            # da li da razbija izjednacene rezultate preciznije (sporije)
    "cache_size": [200, 300, 400],                          # MB memorije za kesiranje (vise = brze za vece datasetove)
    "class_weight": [None, "balanced"],                     # balanced daje vecu tezinu manjinskim klasama
    "coef0": [0.0, 0.1, 0.5, 1.0],                         # slobodan clan u kernel funkciji (utice na poly i sigmoid)
    "decision_function_shape": ["ovo", "ovr"],              # ovo = jedan vs jedan, ovr = jedan vs ostali
    "degree": [3, 4, 5],                                    # stepen polinoma (samo za poly kernel)
    "max_iter": [-1, 100, 200],                             # max iteracija solvera (-1 = bez limita)
    "probability": [True, False],                           # da li da racuna vjerovatnoce predikcija (sporije)
    "shrinking": [True, False],                             # heuristika za ubrzanje treninga
    "tol": [1e-3, 1e-4, 1e-5],                             # tolerancija - kad prestaje trening (manja = preciznije) -> gen. duze traje
}

def random_search(X, y, opcije, n_iter=100):
    best_score = 0
    best_params = None
    best_model = None
    worst_score = 1.0
    worst_params = None
    all_scores = []

    for i in range(n_iter):
        # Nasumicno biramo jednu vrijednost iz svake opcije
        params = {key: random.choice(values) for key, values in opcije.items()}

        # break_ties=True je nekompatibilan sa decision_function_shape='ovo'
        if params["decision_function_shape"] == "ovo":
            params["break_ties"] = False

        # Izdvajamo tt_split jer to nije SVM parametar
        tt_split = params.pop("tt_split")

        # Podjela podataka
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=tt_split, random_state=params["random_state"]
        )

        # Treniranje SVM modela
        model = SVC(**params)
        model.fit(X_train, y_train)

        # Evaluacija
        y_pred = model.predict(X_test)
        score = f1_score(y_test, y_pred, average="weighted")
        all_scores.append(score)

        if score > best_score:
            best_score = score
            best_params = {**params, "tt_split": tt_split}
            best_model = model
            print(f"[Iteracija {i+1}] Novi best F1: {score:.6f} | Params: {best_params}")

        if score < worst_score:
            worst_score = score
            worst_params = {**params, "tt_split": tt_split}

    avg_score = np.mean(all_scores)
    return best_params, best_score, best_model, worst_params, worst_score, avg_score

best_params, best_score, best_model, worst_params, worst_score, avg_score = random_search(X, y, OPCIJE, n_iter=200)
print(f"\nNajbolji parametri: {best_params}")
print(f"Najbolji F1 score: {best_score:.6f}")
print(f"\nNajgori parametri: {worst_params}")
print(f"Najgori F1 score: {worst_score:.6f}")
print(f"\nProsjecni F1 score: {avg_score:.6f}")

## 2. Bat Algorithm (algoritam sismisa) (+1 bod)

Konverzija pseudokoda u Python. sismsi pretrazuju prostor hiperparametara SVM-a kako bi pronasli optimalnu kombinaciju.

Hiperparametri su enkodirani kao kontinualni vektor:
- `dim 0`: tt_split [0.1, 0.5]
- `dim 1`: C [0.1, 10.0]
- `dim 2`: kernel (0=linear, 1=rbf)
- `dim 3`: gamma (0=scale, 1=auto)
- `dim 4`: random_state [0, 49]

In [33]:
# Granice pretrage za svaku dimenziju
# [tt_split, C, kernel, gamma, random_state]
lower_bounds = np.array([0.1, 0.1, 0.0, 0.0, 0])
upper_bounds = np.array([0.5, 10.0, 1.0, 1.0, 49])
n_dims = len(lower_bounds)

# Pydoc docstrings u funkcijama radi lakseg hover ili shif+k umjesto # komentara za svrhu jer zaboravljam :D
def decode_position(x):
    """Dekodira kontinualni vektor pozicije u SVM hiperparametre."""

    tt_split = np.clip(x[0], 0.1, 0.5)

    C = np.clip(x[1], 0.1, 10.0)

    kernel = "rbf" if x[2] >= 0.5 else "linear"

    gamma = "auto" if x[3] >= 0.5 else "scale"

    random_state = int(np.clip(np.round(x[4]), 0, 49))

    return tt_split, C, kernel, gamma, random_state

def objective(x, X, y):
    """Funkcija cilja - vraća negativni F1 score (jer minimiziramo)."""

    tt_split, C, kernel, gamma, rs = decode_position(x)

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=tt_split, random_state=rs
        )

        model = SVC(C=C, kernel=kernel, gamma=gamma)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        score = f1_score(y_test, y_pred, average="weighted")

        return -score  # negativno jer minimiziramo

    except:
        return 0  # najgori mogući score ako model ne uspije

def popravi_granice(x):
    """Vraća poziciju unutar dozvoljenih granica."""

    return np.clip(x, lower_bounds, upper_bounds)

In [34]:
# Bat Algorithm - implementacija prema pseudokodu sa vjezbi

# Parametri algoritma
n_bats = 20          # broj šišmiša u populaciji
n_iter = 50          # broj iteracija
f_min = 0.0          # minimalna frekvencija
f_max = 2.0          # maksimalna frekvencija
alpha = 0.9          # faktor smanjenja glasnoće (0 < alpha < 1)
initial_loudness = 1.0
initial_pulse_rate = 0.5

# Inicijalizacija populacije
x = np.random.uniform(lower_bounds, upper_bounds, (n_bats, n_dims))  # pozicije
v = np.zeros((n_bats, n_dims))                                       # brzine
f = np.zeros(n_bats)                                                  # frekvencije
fitness = np.array([objective(x[i], X, y) for i in range(n_bats)])    # fitness vrijednosti
loudness = np.full(n_bats, initial_loudness)                          # glasnoća
pulse_rate = np.full(n_bats, initial_pulse_rate)                      # pulse rate
all_fitness = []  # za praćenje svih evaluacija

# Pronađi početno najbolje rješenje
best_idx = np.argmin(fitness)
best = x[best_idx].copy()
best_fitness = fitness[best_idx]
worst_fitness = np.max(fitness)

print(f"Početno najbolje rješenje: F1 = {-best_fitness:.6f}")
print(f"Parametri: {decode_position(best)}")
print("-" * 60)

# Glavna petlja algoritma
for t in range(n_iter):
    for i in range(n_bats):  # za svaki bat i:

        beta = np.random.rand()  # beta = random(0, 1)

        f[i] = f_min + (f_max - f_min) * beta  # f[i] = f_min + (f_max - f_min) * beta

        v[i] = v[i] + (x[i] - best) * f[i]  # v[i] = v[i] + (x[i] - best) * f[i]

        x_new = x[i] + v[i]  # x_new = x[i] + v[i]

        # ako random(0,1) > pulse_rate[i]:
        if np.random.rand() > pulse_rate[i]:
            epsilon = np.random.uniform(-1, 1, n_dims)  # epsilon = mali slučajni pomak
            average_loudness = np.mean(loudness)         # prosječna glasnoća svih šišmiša
            x_new = best + epsilon * average_loudness    # x_new = best + epsilon * average_loudness

        x_new = popravi_granice(x_new)  # popravi_granice(x_new)

        fitness_new = objective(x_new, X, y)  # fitness_new = objective(x_new)
        all_fitness.append(-fitness_new)  # čuvamo kao pozitivan F1

        # ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
        if fitness_new < fitness[i] and np.random.rand() < loudness[i]:
            x[i] = x_new              # prihvati novu poziciju
            fitness[i] = fitness_new   # zapamti novi fitness
            loudness[i] = alpha * loudness[i]  # smanji glasnoću

        # Ažuriraj globalno najbolje rješenje
        if fitness[i] < best_fitness:
            best = x[i].copy()
            best_fitness = fitness[i]

        # Ažuriraj najgori fitness
        if fitness_new > worst_fitness:
            worst_fitness = fitness_new

    if (t + 1) % 10 == 0:
        print(f"Iteracija {t+1}/{n_iter} | Najbolji F1: {-best_fitness:.6f} | Najgori F1: {-worst_fitness:.6f} | Avg F1: {np.mean(all_fitness):.6f}")

# Rezultati
print("=" * 60)
tt_split, C, kernel, gamma, rs = decode_position(best)
print(f"Bat Algorithm - Najbolji F1 score: {-best_fitness:.6f}")
print(f"  tt_split     = {tt_split:.4f}")
print(f"  C            = {C:.4f}")
print(f"  kernel       = {kernel}")
print(f"  gamma        = {gamma}")
print(f"  random_state = {rs}")
print(f"\nNajgori F1 score:   {-worst_fitness:.6f}")
print(f"Prosjecni F1 score: {np.mean(all_fitness):.6f}")

Početno najbolje rješenje: F1 = 1.000000
Parametri: (np.float64(0.48632689752949654), np.float64(2.8034933577253884), 'rbf', 'auto', 9)
------------------------------------------------------------
Iteracija 10/50 | Najbolji F1: 1.000000 | Najgori F1: 0.481081 | Avg F1: 0.970709
Iteracija 20/50 | Najbolji F1: 1.000000 | Najgori F1: 0.481081 | Avg F1: 0.975854
Iteracija 30/50 | Najbolji F1: 1.000000 | Najgori F1: 0.481081 | Avg F1: 0.977129
Iteracija 40/50 | Najbolji F1: 1.000000 | Najgori F1: 0.481081 | Avg F1: 0.977675
Iteracija 50/50 | Najbolji F1: 1.000000 | Najgori F1: 0.481081 | Avg F1: 0.978721
Bat Algorithm - Najbolji F1 score: 1.000000
  tt_split     = 0.4863
  C            = 2.8035
  kernel       = rbf
  gamma        = auto
  random_state = 9

Najgori F1 score:   0.481081
Prosjecni F1 score: 0.978721
